# PlantSeg Stage 2 — Session 1 of 3
## Binary Lesion Segmentation: Trainers 1–3 of 6

This is **Session 1** of a three-session split: Session 1 and Session 2 each train 3 models, and Session 3 (run separately, afterward) aggregates all 6 results into the comparison table and figures. All config, data processing, transforms, model code, training loop, and figure generation are reproduced **exactly as-is** from the original per-model notebooks — only `MODEL_NAME` changes between sections.

**This session trains:**
1. UNet (resnet34 encoder)
2. UNet++ (resnet50 encoder)
3. DeepLabV3+ (resnet50 encoder)

Each trainer section writes `s2_result_<MODEL_NAME>.pkl` plus its per-model figures into `/kaggle/working/s2_results/`.

### ⚠️ Required hand-off step
`/kaggle/working` does **not** persist across separate Kaggle sessions. At the end of this notebook, download the 3 `s2_result_*.pkl` files from `/kaggle/working/s2_results/`. Once you also have the 3 from Session 2, combine **all 6** `.pkl` files into one Kaggle Dataset (e.g. named `s2-results`) and use that as input to the **Aggregator session**.

> Internet **ON** is required (encoder pretrained weights). Each model early-stops on val mIoU. Lower `IMG_SIZE` to 384 in the config cell of each section if you hit OOM or want faster training — just keep it consistent across **all 6** trainers (Session 1 and 2) since the aggregator and qualitative figures assume one consistent resize/preprocessing scale.

---
## Trainer 1/6: UNet_resnet34


# PlantSeg Stage 2 — Trainer 1/6: **UNet_resnet34** (binary lesion segmentation)

Trains one model (lesion-vs-background) and saves `s2_result_UNet_resnet34.pkl` + figures to `/kaggle/working/s2_results/`.
Run all 6, collect the `s2_result_*.pkl` into a Kaggle dataset, then run **PlantSeg_S2_aggregate.ipynb**.

> Binary segmentation: feeds Stage-3 severity (lesion area) and Stage-4 XAI faithfulness (attribution vs lesion mask). Internet **ON** (encoder/SegFormer weights). Early-stops on val mIoU so it finishes well within a session. Lower `IMG_SIZE` to 384 if OOM.

In [ ]:
!pip install -q segmentation-models-pytorch albumentations
import os, json, math, time, random, warnings, pickle
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image, ImageDraw
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid", context="talk")
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("Device:",device,"| smp",smp.__version__)

**Config** — paths, hyperparameters, output dirs (identical across all trainers except `MODEL_NAME`):

In [ ]:
# ============ CONFIG ============
MODEL_NAME="UNet_resnet34"            # this notebook trains ONE segmentation model (BINARY lesion vs background)
DATA_ROOT=Path("/kaggle/input/plantseg/plantseg")   # <-- EDIT to your mount
if not (DATA_ROOT/"Metadata.csv").exists():
    for c in [Path("/kaggle/input/plantseg"),Path("plantseg"),Path(".")]:
        if (c/"Metadata.csv").exists(): DATA_ROOT=c; break
assert (DATA_ROOT/"Metadata.csv").exists(), f"Metadata.csv not under {DATA_ROOT} — edit DATA_ROOT."
OUTDIR=Path("/kaggle/working/s2_results"); FIG=OUTDIR/"figures"; MASK=OUTDIR/"masks"
for d in [OUTDIR,FIG,MASK]: d.mkdir(parents=True,exist_ok=True)
SEG_TASK="binary"; NUM_SEG=2; IMG_SIZE=512; NUM_EPOCHS=40; PATIENCE=8; LR=6e-4; WD=1e-4; QUICK_SMOKE=False
print("MODEL:",MODEL_NAME,"| task:",SEG_TASK,"| IMG_SIZE:",IMG_SIZE)

**Index + binary masks** — reads `Metadata.csv`, rasterises COCO-style polygon annotations into binary lesion masks (fill=1 = lesion), builds train/val/test dataframes:

In [ ]:
# ============ INDEX + BINARY MASKS (rasterise polygons; fill=1 = lesion) ============
meta=pd.read_csv(DATA_ROOT/"Metadata.csv",encoding="utf-8-sig")
meta["split"]=meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"]=meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"],axis=1)
def load_anns(split):
    d=json.load(open(DATA_ROOT/f"annotation_{split}.json")); by=defaultdict(list)
    for a in d["annotations"]: by[a["image_id"]].append(a)
    fn={}
    for im in d["images"]:
        polys=[seg for a in by.get(im["id"],[]) for seg in a["segmentation"] if len(seg)>=6]
        fn[im["file_name"]]=(im["width"],im["height"],polys)
    return fn
def rasterize(w,h,polys):
    m=Image.new("L",(w,h),0); dr=ImageDraw.Draw(m)
    for seg in polys: dr.polygon(list(zip(seg[0::2],seg[1::2])),fill=1)
    return m
recs=[]
for split in ["train","val","test"]:
    fn=load_anns(split); od=MASK/split; od.mkdir(parents=True,exist_ok=True)
    for r in meta[meta.split==split].itertuples():
        w,h,polys=fn.get(r.Name,(None,None,[])); mp=od/(Path(r.Name).stem+".png")
        if not mp.exists():
            if w is None:
                with Image.open(r.filepath) as i0: w,h=i0.size
            rasterize(w,h,polys).save(mp)
        recs.append({"name":r.Name,"filepath":str(r.filepath),"mask":str(mp),"split":split,
                     "disease":r.Disease,"host":r.Plant})
df=pd.DataFrame(recs)
if "Mask ratio" in meta.columns:
    df["mask_ratio"]=df["name"].map(meta.set_index("Name")["Mask ratio"]).astype(float)
train_df=df[df.split=="train"].reset_index(drop=True); val_df=df[df.split=="val"].reset_index(drop=True); test_df=df[df.split=="test"].reset_index(drop=True)
if QUICK_SMOKE:
    train_df=train_df.head(40); val_df=val_df.head(40); test_df=test_df.head(40); NUM_EPOCHS=1
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)} | binary masks ready")

**Transforms / Dataset** — Albumentations train/eval pipelines and the `SegDS` `Dataset`/`DataLoader` builder:

In [ ]:
# ============ TRANSFORMS / DATASET ============
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
train_tf=A.Compose([A.RandomResizedCrop(size=(IMG_SIZE,IMG_SIZE),scale=(0.6,1.0)),A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),A.Rotate(limit=20,p=0.5),A.RandomBrightnessContrast(0.2,0.2,p=0.5),A.Normalize(MEAN,STD),ToTensorV2()])
eval_tf=A.Compose([A.Resize(IMG_SIZE,IMG_SIZE),A.Normalize(MEAN,STD),ToTensorV2()])
class SegDS(Dataset):
    def __init__(s,d,t): s.df=d.reset_index(drop=True); s.t=t
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]; img=np.array(Image.open(r.filepath).convert("RGB")); m=np.array(Image.open(r["mask"]))
        a=s.t(image=img,mask=m); return a["image"], a["mask"].long()
def loaders(bs):
    return (DataLoader(SegDS(train_df,train_tf),batch_size=bs,shuffle=True,num_workers=2,pin_memory=True,drop_last=True),
            DataLoader(SegDS(val_df,eval_tf),batch_size=bs,shuffle=False,num_workers=2,pin_memory=True),
            DataLoader(SegDS(test_df,eval_tf),batch_size=bs,shuffle=False,num_workers=2,pin_memory=True))

**Model** — unified wrapper (`SegNet`) so SMP architectures and the HuggingFace SegFormer share one forward-pass interface returning full-resolution logits:

In [ ]:
# ============ MODEL (unified full-res logits) ============
from transformers import SegformerForSemanticSegmentation
SMP_SPECS={"UNet_resnet34":(smp.Unet,"resnet34"),"UNetPP_resnet50":(smp.UnetPlusPlus,"resnet50"),
    "DeepLabV3Plus_resnet50":(smp.DeepLabV3Plus,"resnet50"),
    "DeepLabV3Plus_efficientnet-b3":(smp.DeepLabV3Plus,"efficientnet-b3"),
    "FPN_resnet50":(smp.FPN,"resnet50"),"PSPNet_resnet50":(smp.PSPNet,"resnet50")}
SEGFORMER_HF={"SegFormer_b0":"nvidia/mit-b0","SegFormer_b2":"nvidia/mit-b2"}
class SegNet(nn.Module):
    def __init__(s,name,nc):
        super().__init__()
        if name in SMP_SPECS:
            arch,enc=SMP_SPECS[name]; s.net=arch(encoder_name=enc,encoder_weights="imagenet",in_channels=3,classes=nc); s.kind="smp"
        else:
            s.net=SegformerForSemanticSegmentation.from_pretrained(SEGFORMER_HF[name],num_labels=nc,ignore_mismatched_sizes=True); s.kind="hf"
    def forward(s,x):
        if s.kind=="smp": return s.net(x)
        lg=s.net(pixel_values=x).logits; return F.interpolate(lg,size=x.shape[-2:],mode="bilinear",align_corners=False)
def bs_for(name): return 6 if ("b2" in name or "b4" in name) else 8

**Metrics / Loss / Train** — confusion-matrix based IoU/Dice, Dice+CE loss, the early-stopping training loop, deployment benchmarking (params, checkpoint size, CPU/GPU latency), and per-image foreground IoU:

In [ ]:
# ============ METRICS / LOSS / TRAIN (early-stop on val mIoU) ============
class Conf:
    def __init__(s,n): s.n=n; s.cm=np.zeros((n,n),np.int64)
    def add(s,gt,pr):
        k=(gt>=0)&(gt<s.n); s.cm+=np.bincount(s.n*gt[k].astype(int)+pr[k],minlength=s.n**2).reshape(s.n,s.n)
    def scores(s):
        cm=s.cm.astype(float); tp=np.diag(cm); iou=tp/(cm.sum(1)+cm.sum(0)-tp+1e-9); dice=2*tp/(cm.sum(1)+cm.sum(0)+1e-9)
        pr=cm.sum(1)>0
        return {"mIoU":float(np.nanmean(iou[pr])),"mDice":float(np.nanmean(dice[pr])),
                "pixel_acc":float(tp.sum()/(cm.sum()+1e-9)),"fg_IoU":float(iou[1]),"fg_Dice":float(dice[1])}, iou
def dice_ce(lg,t,n):
    ce=F.cross_entropy(lg,t); p=F.softmax(lg,1); oh=F.one_hot(t,n).permute(0,3,1,2).float()
    d=1-((2*(p*oh).sum((0,2,3))+1)/(p.sum((0,2,3))+oh.sum((0,2,3))+1)).mean(); return ce+d
def evaluate(model,loader,n):
    model.eval(); cf=Conf(n)
    with torch.no_grad():
        for x,y in loader:
            pr=model(x.to(device)).argmax(1).cpu().numpy(); cf.add(y.numpy().ravel(),pr.ravel())
    return cf
def train_one(name,n=NUM_SEG,epochs=NUM_EPOCHS,patience=PATIENCE):
    model=SegNet(name,n).to(device); bs=bs_for(name); tl,vl,te=loaders(bs)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    scaler=torch.cuda.amp.GradScaler(enabled=device.type=="cuda")
    hist={"train_loss":[],"val_mIoU":[],"val_fgIoU":[]}; best=-1; best_state=None; bad=0
    for ep in range(epochs):
        model.train(); run=nb=0
        for x,y in tl:
            x,y=x.to(device),y.to(device); opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=device.type=="cuda"): loss=dice_ce(model(x),y,n)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); run+=loss.item(); nb+=1
        sch.step(); sc,_=evaluate(model,vl,n).scores()
        hist["train_loss"].append(run/max(nb,1)); hist["val_mIoU"].append(sc["mIoU"]); hist["val_fgIoU"].append(sc["fg_IoU"])
        print(f"[{name}] ep{ep+1}/{epochs} loss={run/max(nb,1):.3f} val_mIoU={sc['mIoU']:.4f} val_fgIoU={sc['fg_IoU']:.4f}")
        if sc["mIoU"]>best+1e-4: best=sc["mIoU"]; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=patience: print(f"  early stop @ ep{ep+1}"); break
    model.load_state_dict(best_state); torch.save(best_state,OUTDIR/f"{name}.pt")
    return model,hist,bs
def deployment(model,img=IMG_SIZE,nw=3,it=15):
    p=sum(x.numel() for x in model.parameters()); tmp=OUTDIR/"_t.pt"; torch.save(model.state_dict(),tmp)
    sz=os.path.getsize(tmp)/1e6; os.remove(tmp); d=torch.randn(1,3,img,img)
    def t(dev):
        m=model.to(dev).eval(); dd=d.to(dev)
        with torch.no_grad():
            for _ in range(nw): m(dd)
            if dev.type=="cuda": torch.cuda.synchronize()
            t0=time.time()
            for _ in range(it): m(dd)
            if dev.type=="cuda": torch.cuda.synchronize()
            return (time.time()-t0)/it*1000
    lg=t(device) if device.type=="cuda" else np.nan; lc=t(torch.device("cpu")); model.to(device)
    return {"params_M":p/1e6,"ckpt_MB":sz,"latency_cpu_ms":lc,"latency_gpu_ms":lg}
def per_image_fg_iou(model):
    model.eval(); v=[]
    with torch.no_grad():
        for r in test_df.itertuples():
            img=np.array(Image.open(r.filepath).convert("RGB")); g=np.array(Image.open(r.mask))
            a=eval_tf(image=img,mask=g); pr=model(a["image"].unsqueeze(0).to(device)).argmax(1)[0].cpu().numpy()
            gg=a["mask"].numpy()>0; pp=pr>0; inter=(gg&pp).sum(); uni=(gg|pp).sum(); v.append(inter/uni if uni>0 else 1.0)
    return np.array(v)

**Run + Save** — trains the model, evaluates on the test split, benchmarks deployment cost, and pickles the full result bundle to `s2_result_<MODEL_NAME>.pkl`:

In [ ]:
# ============ RUN + SAVE ============
t0=time.time(); model,hist,bs=train_one(MODEL_NAME); ttime=time.time()-t0
cf=evaluate(model,loaders(bs)[2],NUM_SEG); sc,iou=cf.scores(); dep=deployment(model)
pifg=per_image_fg_iou(model)
res={"model_name":MODEL_NAME,"seg_task":SEG_TASK,"history":hist,"test_scores":sc,"iou":iou.tolist(),
     "deployment":dep,"per_image_fg_iou":pifg,"img_size":IMG_SIZE,"train_time_sec":ttime}
pickle.dump(res,open(OUTDIR/f"s2_result_{MODEL_NAME}.pkl","wb"))
print("SAVED",OUTDIR/f"s2_result_{MODEL_NAME}.pkl")
print("TEST:",{k:round(v,4) for k,v in sc.items()},"| params",round(dep['params_M'],1),"M | gpu",round(dep['latency_gpu_ms'],1),"ms")

**Per-model figures** — training curve (loss + val mIoU/fg-IoU) and qualitative input/ground-truth/prediction grid, saved to `figures/`:

In [ ]:
# ============ PER-MODEL FIGURES ============
ep=range(1,len(hist["train_loss"])+1)
fig,ax=plt.subplots(figsize=(8,4.5)); ax.plot(ep,hist["train_loss"],"--",label="train loss")
a2=ax.twinx(); a2.plot(ep,hist["val_mIoU"],color="green",label="val mIoU"); a2.plot(ep,hist["val_fgIoU"],color="orange",label="val fg-IoU")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); a2.set_ylabel("IoU"); ax.set_title(f"{MODEL_NAME} — training")
plt.tight_layout(); plt.savefig(FIG/f"{MODEL_NAME}_training.png",dpi=200,bbox_inches="tight"); plt.show()

samp=test_df.sample(min(5,len(test_df)),random_state=SEED).reset_index(drop=True)
fig,axes=plt.subplots(len(samp),3,figsize=(10,3.2*len(samp))); axes=np.atleast_2d(axes)
for i,r in samp.iterrows():
    img=Image.open(r.filepath).convert("RGB"); gt=np.array(Image.open(r["mask"]))>0
    a=eval_tf(image=np.array(img),mask=np.array(Image.open(r["mask"])))
    with torch.no_grad(): pr=model(a["image"].unsqueeze(0).to(device)).argmax(1)[0].cpu().numpy()>0
    axes[i,0].imshow(img.resize((IMG_SIZE,IMG_SIZE))); axes[i,1].imshow(gt,cmap="gray"); axes[i,2].imshow(pr,cmap="gray")
    if i==0:
        for j,t_ in enumerate(["input","ground truth","prediction"]): axes[i,j].set_title(t_)
    axes[i,0].set_ylabel(r.disease,fontsize=8)
    for a_ in axes[i]: a_.set_xticks([]); a_.set_yticks([])
plt.suptitle(f"{MODEL_NAME} — qualitative (lesion mask)"); plt.tight_layout()
plt.savefig(FIG/f"{MODEL_NAME}_qualitative.png",dpi=200,bbox_inches="tight"); plt.show()

---
## Trainer 2/6: UNetPP_resnet50


# PlantSeg Stage 2 — Trainer 2/6: **UNetPP_resnet50** (binary lesion segmentation)

Trains one model (lesion-vs-background) and saves `s2_result_UNetPP_resnet50.pkl` + figures to `/kaggle/working/s2_results/`.
Run all 6, collect the `s2_result_*.pkl` into a Kaggle dataset, then run **PlantSeg_S2_aggregate.ipynb**.

> Binary segmentation: feeds Stage-3 severity (lesion area) and Stage-4 XAI faithfulness (attribution vs lesion mask). Internet **ON** (encoder/SegFormer weights). Early-stops on val mIoU so it finishes well within a session. Lower `IMG_SIZE` to 384 if OOM.

In [ ]:
!pip install -q segmentation-models-pytorch albumentations
import os, json, math, time, random, warnings, pickle
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image, ImageDraw
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid", context="talk")
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("Device:",device,"| smp",smp.__version__)

**Config** — paths, hyperparameters, output dirs (identical across all trainers except `MODEL_NAME`):

In [ ]:
# ============ CONFIG ============
MODEL_NAME="UNetPP_resnet50"            # this notebook trains ONE segmentation model (BINARY lesion vs background)
DATA_ROOT=Path("/kaggle/input/plantseg/plantseg")   # <-- EDIT to your mount
if not (DATA_ROOT/"Metadata.csv").exists():
    for c in [Path("/kaggle/input/plantseg"),Path("plantseg"),Path(".")]:
        if (c/"Metadata.csv").exists(): DATA_ROOT=c; break
assert (DATA_ROOT/"Metadata.csv").exists(), f"Metadata.csv not under {DATA_ROOT} — edit DATA_ROOT."
OUTDIR=Path("/kaggle/working/s2_results"); FIG=OUTDIR/"figures"; MASK=OUTDIR/"masks"
for d in [OUTDIR,FIG,MASK]: d.mkdir(parents=True,exist_ok=True)
SEG_TASK="binary"; NUM_SEG=2; IMG_SIZE=512; NUM_EPOCHS=40; PATIENCE=8; LR=6e-4; WD=1e-4; QUICK_SMOKE=False
print("MODEL:",MODEL_NAME,"| task:",SEG_TASK,"| IMG_SIZE:",IMG_SIZE)

**Index + binary masks** — reads `Metadata.csv`, rasterises COCO-style polygon annotations into binary lesion masks (fill=1 = lesion), builds train/val/test dataframes:

In [ ]:
# ============ INDEX + BINARY MASKS (rasterise polygons; fill=1 = lesion) ============
meta=pd.read_csv(DATA_ROOT/"Metadata.csv",encoding="utf-8-sig")
meta["split"]=meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"]=meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"],axis=1)
def load_anns(split):
    d=json.load(open(DATA_ROOT/f"annotation_{split}.json")); by=defaultdict(list)
    for a in d["annotations"]: by[a["image_id"]].append(a)
    fn={}
    for im in d["images"]:
        polys=[seg for a in by.get(im["id"],[]) for seg in a["segmentation"] if len(seg)>=6]
        fn[im["file_name"]]=(im["width"],im["height"],polys)
    return fn
def rasterize(w,h,polys):
    m=Image.new("L",(w,h),0); dr=ImageDraw.Draw(m)
    for seg in polys: dr.polygon(list(zip(seg[0::2],seg[1::2])),fill=1)
    return m
recs=[]
for split in ["train","val","test"]:
    fn=load_anns(split); od=MASK/split; od.mkdir(parents=True,exist_ok=True)
    for r in meta[meta.split==split].itertuples():
        w,h,polys=fn.get(r.Name,(None,None,[])); mp=od/(Path(r.Name).stem+".png")
        if not mp.exists():
            if w is None:
                with Image.open(r.filepath) as i0: w,h=i0.size
            rasterize(w,h,polys).save(mp)
        recs.append({"name":r.Name,"filepath":str(r.filepath),"mask":str(mp),"split":split,
                     "disease":r.Disease,"host":r.Plant})
df=pd.DataFrame(recs)
if "Mask ratio" in meta.columns:
    df["mask_ratio"]=df["name"].map(meta.set_index("Name")["Mask ratio"]).astype(float)
train_df=df[df.split=="train"].reset_index(drop=True); val_df=df[df.split=="val"].reset_index(drop=True); test_df=df[df.split=="test"].reset_index(drop=True)
if QUICK_SMOKE:
    train_df=train_df.head(40); val_df=val_df.head(40); test_df=test_df.head(40); NUM_EPOCHS=1
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)} | binary masks ready")

**Transforms / Dataset** — Albumentations train/eval pipelines and the `SegDS` `Dataset`/`DataLoader` builder:

In [ ]:
# ============ TRANSFORMS / DATASET ============
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
train_tf=A.Compose([A.RandomResizedCrop(size=(IMG_SIZE,IMG_SIZE),scale=(0.6,1.0)),A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),A.Rotate(limit=20,p=0.5),A.RandomBrightnessContrast(0.2,0.2,p=0.5),A.Normalize(MEAN,STD),ToTensorV2()])
eval_tf=A.Compose([A.Resize(IMG_SIZE,IMG_SIZE),A.Normalize(MEAN,STD),ToTensorV2()])
class SegDS(Dataset):
    def __init__(s,d,t): s.df=d.reset_index(drop=True); s.t=t
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]; img=np.array(Image.open(r.filepath).convert("RGB")); m=np.array(Image.open(r["mask"]))
        a=s.t(image=img,mask=m); return a["image"], a["mask"].long()
def loaders(bs):
    return (DataLoader(SegDS(train_df,train_tf),batch_size=bs,shuffle=True,num_workers=2,pin_memory=True,drop_last=True),
            DataLoader(SegDS(val_df,eval_tf),batch_size=bs,shuffle=False,num_workers=2,pin_memory=True),
            DataLoader(SegDS(test_df,eval_tf),batch_size=bs,shuffle=False,num_workers=2,pin_memory=True))

**Model** — unified wrapper (`SegNet`) so SMP architectures and the HuggingFace SegFormer share one forward-pass interface returning full-resolution logits:

In [ ]:
# ============ MODEL (unified full-res logits) ============
from transformers import SegformerForSemanticSegmentation
SMP_SPECS={"UNet_resnet34":(smp.Unet,"resnet34"),"UNetPP_resnet50":(smp.UnetPlusPlus,"resnet50"),
    "DeepLabV3Plus_resnet50":(smp.DeepLabV3Plus,"resnet50"),
    "DeepLabV3Plus_efficientnet-b3":(smp.DeepLabV3Plus,"efficientnet-b3"),
    "FPN_resnet50":(smp.FPN,"resnet50"),"PSPNet_resnet50":(smp.PSPNet,"resnet50")}
SEGFORMER_HF={"SegFormer_b0":"nvidia/mit-b0","SegFormer_b2":"nvidia/mit-b2"}
class SegNet(nn.Module):
    def __init__(s,name,nc):
        super().__init__()
        if name in SMP_SPECS:
            arch,enc=SMP_SPECS[name]; s.net=arch(encoder_name=enc,encoder_weights="imagenet",in_channels=3,classes=nc); s.kind="smp"
        else:
            s.net=SegformerForSemanticSegmentation.from_pretrained(SEGFORMER_HF[name],num_labels=nc,ignore_mismatched_sizes=True); s.kind="hf"
    def forward(s,x):
        if s.kind=="smp": return s.net(x)
        lg=s.net(pixel_values=x).logits; return F.interpolate(lg,size=x.shape[-2:],mode="bilinear",align_corners=False)
def bs_for(name): return 6 if ("b2" in name or "b4" in name) else 8

**Metrics / Loss / Train** — confusion-matrix based IoU/Dice, Dice+CE loss, the early-stopping training loop, deployment benchmarking (params, checkpoint size, CPU/GPU latency), and per-image foreground IoU:

In [ ]:
# ============ METRICS / LOSS / TRAIN (early-stop on val mIoU) ============
class Conf:
    def __init__(s,n): s.n=n; s.cm=np.zeros((n,n),np.int64)
    def add(s,gt,pr):
        k=(gt>=0)&(gt<s.n); s.cm+=np.bincount(s.n*gt[k].astype(int)+pr[k],minlength=s.n**2).reshape(s.n,s.n)
    def scores(s):
        cm=s.cm.astype(float); tp=np.diag(cm); iou=tp/(cm.sum(1)+cm.sum(0)-tp+1e-9); dice=2*tp/(cm.sum(1)+cm.sum(0)+1e-9)
        pr=cm.sum(1)>0
        return {"mIoU":float(np.nanmean(iou[pr])),"mDice":float(np.nanmean(dice[pr])),
                "pixel_acc":float(tp.sum()/(cm.sum()+1e-9)),"fg_IoU":float(iou[1]),"fg_Dice":float(dice[1])}, iou
def dice_ce(lg,t,n):
    ce=F.cross_entropy(lg,t); p=F.softmax(lg,1); oh=F.one_hot(t,n).permute(0,3,1,2).float()
    d=1-((2*(p*oh).sum((0,2,3))+1)/(p.sum((0,2,3))+oh.sum((0,2,3))+1)).mean(); return ce+d
def evaluate(model,loader,n):
    model.eval(); cf=Conf(n)
    with torch.no_grad():
        for x,y in loader:
            pr=model(x.to(device)).argmax(1).cpu().numpy(); cf.add(y.numpy().ravel(),pr.ravel())
    return cf
def train_one(name,n=NUM_SEG,epochs=NUM_EPOCHS,patience=PATIENCE):
    model=SegNet(name,n).to(device); bs=bs_for(name); tl,vl,te=loaders(bs)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    scaler=torch.cuda.amp.GradScaler(enabled=device.type=="cuda")
    hist={"train_loss":[],"val_mIoU":[],"val_fgIoU":[]}; best=-1; best_state=None; bad=0
    for ep in range(epochs):
        model.train(); run=nb=0
        for x,y in tl:
            x,y=x.to(device),y.to(device); opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=device.type=="cuda"): loss=dice_ce(model(x),y,n)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); run+=loss.item(); nb+=1
        sch.step(); sc,_=evaluate(model,vl,n).scores()
        hist["train_loss"].append(run/max(nb,1)); hist["val_mIoU"].append(sc["mIoU"]); hist["val_fgIoU"].append(sc["fg_IoU"])
        print(f"[{name}] ep{ep+1}/{epochs} loss={run/max(nb,1):.3f} val_mIoU={sc['mIoU']:.4f} val_fgIoU={sc['fg_IoU']:.4f}")
        if sc["mIoU"]>best+1e-4: best=sc["mIoU"]; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=patience: print(f"  early stop @ ep{ep+1}"); break
    model.load_state_dict(best_state); torch.save(best_state,OUTDIR/f"{name}.pt")
    return model,hist,bs
def deployment(model,img=IMG_SIZE,nw=3,it=15):
    p=sum(x.numel() for x in model.parameters()); tmp=OUTDIR/"_t.pt"; torch.save(model.state_dict(),tmp)
    sz=os.path.getsize(tmp)/1e6; os.remove(tmp); d=torch.randn(1,3,img,img)
    def t(dev):
        m=model.to(dev).eval(); dd=d.to(dev)
        with torch.no_grad():
            for _ in range(nw): m(dd)
            if dev.type=="cuda": torch.cuda.synchronize()
            t0=time.time()
            for _ in range(it): m(dd)
            if dev.type=="cuda": torch.cuda.synchronize()
            return (time.time()-t0)/it*1000
    lg=t(device) if device.type=="cuda" else np.nan; lc=t(torch.device("cpu")); model.to(device)
    return {"params_M":p/1e6,"ckpt_MB":sz,"latency_cpu_ms":lc,"latency_gpu_ms":lg}
def per_image_fg_iou(model):
    model.eval(); v=[]
    with torch.no_grad():
        for r in test_df.itertuples():
            img=np.array(Image.open(r.filepath).convert("RGB")); g=np.array(Image.open(r.mask))
            a=eval_tf(image=img,mask=g); pr=model(a["image"].unsqueeze(0).to(device)).argmax(1)[0].cpu().numpy()
            gg=a["mask"].numpy()>0; pp=pr>0; inter=(gg&pp).sum(); uni=(gg|pp).sum(); v.append(inter/uni if uni>0 else 1.0)
    return np.array(v)

**Run + Save** — trains the model, evaluates on the test split, benchmarks deployment cost, and pickles the full result bundle to `s2_result_<MODEL_NAME>.pkl`:

In [ ]:
# ============ RUN + SAVE ============
t0=time.time(); model,hist,bs=train_one(MODEL_NAME); ttime=time.time()-t0
cf=evaluate(model,loaders(bs)[2],NUM_SEG); sc,iou=cf.scores(); dep=deployment(model)
pifg=per_image_fg_iou(model)
res={"model_name":MODEL_NAME,"seg_task":SEG_TASK,"history":hist,"test_scores":sc,"iou":iou.tolist(),
     "deployment":dep,"per_image_fg_iou":pifg,"img_size":IMG_SIZE,"train_time_sec":ttime}
pickle.dump(res,open(OUTDIR/f"s2_result_{MODEL_NAME}.pkl","wb"))
print("SAVED",OUTDIR/f"s2_result_{MODEL_NAME}.pkl")
print("TEST:",{k:round(v,4) for k,v in sc.items()},"| params",round(dep['params_M'],1),"M | gpu",round(dep['latency_gpu_ms'],1),"ms")

**Per-model figures** — training curve (loss + val mIoU/fg-IoU) and qualitative input/ground-truth/prediction grid, saved to `figures/`:

In [ ]:
# ============ PER-MODEL FIGURES ============
ep=range(1,len(hist["train_loss"])+1)
fig,ax=plt.subplots(figsize=(8,4.5)); ax.plot(ep,hist["train_loss"],"--",label="train loss")
a2=ax.twinx(); a2.plot(ep,hist["val_mIoU"],color="green",label="val mIoU"); a2.plot(ep,hist["val_fgIoU"],color="orange",label="val fg-IoU")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); a2.set_ylabel("IoU"); ax.set_title(f"{MODEL_NAME} — training")
plt.tight_layout(); plt.savefig(FIG/f"{MODEL_NAME}_training.png",dpi=200,bbox_inches="tight"); plt.show()

samp=test_df.sample(min(5,len(test_df)),random_state=SEED).reset_index(drop=True)
fig,axes=plt.subplots(len(samp),3,figsize=(10,3.2*len(samp))); axes=np.atleast_2d(axes)
for i,r in samp.iterrows():
    img=Image.open(r.filepath).convert("RGB"); gt=np.array(Image.open(r["mask"]))>0
    a=eval_tf(image=np.array(img),mask=np.array(Image.open(r["mask"])))
    with torch.no_grad(): pr=model(a["image"].unsqueeze(0).to(device)).argmax(1)[0].cpu().numpy()>0
    axes[i,0].imshow(img.resize((IMG_SIZE,IMG_SIZE))); axes[i,1].imshow(gt,cmap="gray"); axes[i,2].imshow(pr,cmap="gray")
    if i==0:
        for j,t_ in enumerate(["input","ground truth","prediction"]): axes[i,j].set_title(t_)
    axes[i,0].set_ylabel(r.disease,fontsize=8)
    for a_ in axes[i]: a_.set_xticks([]); a_.set_yticks([])
plt.suptitle(f"{MODEL_NAME} — qualitative (lesion mask)"); plt.tight_layout()
plt.savefig(FIG/f"{MODEL_NAME}_qualitative.png",dpi=200,bbox_inches="tight"); plt.show()

---
## Trainer 3/6: DeepLabV3Plus_resnet50


# PlantSeg Stage 2 — Trainer 3/6: **DeepLabV3Plus_resnet50** (binary lesion segmentation)

Trains one model (lesion-vs-background) and saves `s2_result_DeepLabV3Plus_resnet50.pkl` + figures to `/kaggle/working/s2_results/`.
Run all 6, collect the `s2_result_*.pkl` into a Kaggle dataset, then run **PlantSeg_S2_aggregate.ipynb**.

> Binary segmentation: feeds Stage-3 severity (lesion area) and Stage-4 XAI faithfulness (attribution vs lesion mask). Internet **ON** (encoder/SegFormer weights). Early-stops on val mIoU so it finishes well within a session. Lower `IMG_SIZE` to 384 if OOM.

In [ ]:
!pip install -q segmentation-models-pytorch albumentations
import os, json, math, time, random, warnings, pickle
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image, ImageDraw
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid", context="talk")
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("Device:",device,"| smp",smp.__version__)

**Config** — paths, hyperparameters, output dirs (identical across all trainers except `MODEL_NAME`):

In [ ]:
# ============ CONFIG ============
MODEL_NAME="DeepLabV3Plus_resnet50"            # this notebook trains ONE segmentation model (BINARY lesion vs background)
DATA_ROOT=Path("/kaggle/input/plantseg/plantseg")   # <-- EDIT to your mount
if not (DATA_ROOT/"Metadata.csv").exists():
    for c in [Path("/kaggle/input/plantseg"),Path("plantseg"),Path(".")]:
        if (c/"Metadata.csv").exists(): DATA_ROOT=c; break
assert (DATA_ROOT/"Metadata.csv").exists(), f"Metadata.csv not under {DATA_ROOT} — edit DATA_ROOT."
OUTDIR=Path("/kaggle/working/s2_results"); FIG=OUTDIR/"figures"; MASK=OUTDIR/"masks"
for d in [OUTDIR,FIG,MASK]: d.mkdir(parents=True,exist_ok=True)
SEG_TASK="binary"; NUM_SEG=2; IMG_SIZE=512; NUM_EPOCHS=40; PATIENCE=8; LR=6e-4; WD=1e-4; QUICK_SMOKE=False
print("MODEL:",MODEL_NAME,"| task:",SEG_TASK,"| IMG_SIZE:",IMG_SIZE)

**Index + binary masks** — reads `Metadata.csv`, rasterises COCO-style polygon annotations into binary lesion masks (fill=1 = lesion), builds train/val/test dataframes:

In [ ]:
# ============ INDEX + BINARY MASKS (rasterise polygons; fill=1 = lesion) ============
meta=pd.read_csv(DATA_ROOT/"Metadata.csv",encoding="utf-8-sig")
meta["split"]=meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"]=meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"],axis=1)
def load_anns(split):
    d=json.load(open(DATA_ROOT/f"annotation_{split}.json")); by=defaultdict(list)
    for a in d["annotations"]: by[a["image_id"]].append(a)
    fn={}
    for im in d["images"]:
        polys=[seg for a in by.get(im["id"],[]) for seg in a["segmentation"] if len(seg)>=6]
        fn[im["file_name"]]=(im["width"],im["height"],polys)
    return fn
def rasterize(w,h,polys):
    m=Image.new("L",(w,h),0); dr=ImageDraw.Draw(m)
    for seg in polys: dr.polygon(list(zip(seg[0::2],seg[1::2])),fill=1)
    return m
recs=[]
for split in ["train","val","test"]:
    fn=load_anns(split); od=MASK/split; od.mkdir(parents=True,exist_ok=True)
    for r in meta[meta.split==split].itertuples():
        w,h,polys=fn.get(r.Name,(None,None,[])); mp=od/(Path(r.Name).stem+".png")
        if not mp.exists():
            if w is None:
                with Image.open(r.filepath) as i0: w,h=i0.size
            rasterize(w,h,polys).save(mp)
        recs.append({"name":r.Name,"filepath":str(r.filepath),"mask":str(mp),"split":split,
                     "disease":r.Disease,"host":r.Plant})
df=pd.DataFrame(recs)
if "Mask ratio" in meta.columns:
    df["mask_ratio"]=df["name"].map(meta.set_index("Name")["Mask ratio"]).astype(float)
train_df=df[df.split=="train"].reset_index(drop=True); val_df=df[df.split=="val"].reset_index(drop=True); test_df=df[df.split=="test"].reset_index(drop=True)
if QUICK_SMOKE:
    train_df=train_df.head(40); val_df=val_df.head(40); test_df=test_df.head(40); NUM_EPOCHS=1
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)} | binary masks ready")

**Transforms / Dataset** — Albumentations train/eval pipelines and the `SegDS` `Dataset`/`DataLoader` builder:

In [ ]:
# ============ TRANSFORMS / DATASET ============
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
train_tf=A.Compose([A.RandomResizedCrop(size=(IMG_SIZE,IMG_SIZE),scale=(0.6,1.0)),A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),A.Rotate(limit=20,p=0.5),A.RandomBrightnessContrast(0.2,0.2,p=0.5),A.Normalize(MEAN,STD),ToTensorV2()])
eval_tf=A.Compose([A.Resize(IMG_SIZE,IMG_SIZE),A.Normalize(MEAN,STD),ToTensorV2()])
class SegDS(Dataset):
    def __init__(s,d,t): s.df=d.reset_index(drop=True); s.t=t
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]; img=np.array(Image.open(r.filepath).convert("RGB")); m=np.array(Image.open(r["mask"]))
        a=s.t(image=img,mask=m); return a["image"], a["mask"].long()
def loaders(bs):
    return (DataLoader(SegDS(train_df,train_tf),batch_size=bs,shuffle=True,num_workers=2,pin_memory=True,drop_last=True),
            DataLoader(SegDS(val_df,eval_tf),batch_size=bs,shuffle=False,num_workers=2,pin_memory=True),
            DataLoader(SegDS(test_df,eval_tf),batch_size=bs,shuffle=False,num_workers=2,pin_memory=True))

**Model** — unified wrapper (`SegNet`) so SMP architectures and the HuggingFace SegFormer share one forward-pass interface returning full-resolution logits:

In [ ]:
# ============ MODEL (unified full-res logits) ============
from transformers import SegformerForSemanticSegmentation
SMP_SPECS={"UNet_resnet34":(smp.Unet,"resnet34"),"UNetPP_resnet50":(smp.UnetPlusPlus,"resnet50"),
    "DeepLabV3Plus_resnet50":(smp.DeepLabV3Plus,"resnet50"),
    "DeepLabV3Plus_efficientnet-b3":(smp.DeepLabV3Plus,"efficientnet-b3"),
    "FPN_resnet50":(smp.FPN,"resnet50"),"PSPNet_resnet50":(smp.PSPNet,"resnet50")}
SEGFORMER_HF={"SegFormer_b0":"nvidia/mit-b0","SegFormer_b2":"nvidia/mit-b2"}
class SegNet(nn.Module):
    def __init__(s,name,nc):
        super().__init__()
        if name in SMP_SPECS:
            arch,enc=SMP_SPECS[name]; s.net=arch(encoder_name=enc,encoder_weights="imagenet",in_channels=3,classes=nc); s.kind="smp"
        else:
            s.net=SegformerForSemanticSegmentation.from_pretrained(SEGFORMER_HF[name],num_labels=nc,ignore_mismatched_sizes=True); s.kind="hf"
    def forward(s,x):
        if s.kind=="smp": return s.net(x)
        lg=s.net(pixel_values=x).logits; return F.interpolate(lg,size=x.shape[-2:],mode="bilinear",align_corners=False)
def bs_for(name): return 6 if ("b2" in name or "b4" in name) else 8

**Metrics / Loss / Train** — confusion-matrix based IoU/Dice, Dice+CE loss, the early-stopping training loop, deployment benchmarking (params, checkpoint size, CPU/GPU latency), and per-image foreground IoU:

In [ ]:
# ============ METRICS / LOSS / TRAIN (early-stop on val mIoU) ============
class Conf:
    def __init__(s,n): s.n=n; s.cm=np.zeros((n,n),np.int64)
    def add(s,gt,pr):
        k=(gt>=0)&(gt<s.n); s.cm+=np.bincount(s.n*gt[k].astype(int)+pr[k],minlength=s.n**2).reshape(s.n,s.n)
    def scores(s):
        cm=s.cm.astype(float); tp=np.diag(cm); iou=tp/(cm.sum(1)+cm.sum(0)-tp+1e-9); dice=2*tp/(cm.sum(1)+cm.sum(0)+1e-9)
        pr=cm.sum(1)>0
        return {"mIoU":float(np.nanmean(iou[pr])),"mDice":float(np.nanmean(dice[pr])),
                "pixel_acc":float(tp.sum()/(cm.sum()+1e-9)),"fg_IoU":float(iou[1]),"fg_Dice":float(dice[1])}, iou
def dice_ce(lg,t,n):
    ce=F.cross_entropy(lg,t); p=F.softmax(lg,1); oh=F.one_hot(t,n).permute(0,3,1,2).float()
    d=1-((2*(p*oh).sum((0,2,3))+1)/(p.sum((0,2,3))+oh.sum((0,2,3))+1)).mean(); return ce+d
def evaluate(model,loader,n):
    model.eval(); cf=Conf(n)
    with torch.no_grad():
        for x,y in loader:
            pr=model(x.to(device)).argmax(1).cpu().numpy(); cf.add(y.numpy().ravel(),pr.ravel())
    return cf
def train_one(name,n=NUM_SEG,epochs=NUM_EPOCHS,patience=PATIENCE):
    model=SegNet(name,n).to(device); bs=bs_for(name); tl,vl,te=loaders(bs)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    scaler=torch.cuda.amp.GradScaler(enabled=device.type=="cuda")
    hist={"train_loss":[],"val_mIoU":[],"val_fgIoU":[]}; best=-1; best_state=None; bad=0
    for ep in range(epochs):
        model.train(); run=nb=0
        for x,y in tl:
            x,y=x.to(device),y.to(device); opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=device.type=="cuda"): loss=dice_ce(model(x),y,n)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); run+=loss.item(); nb+=1
        sch.step(); sc,_=evaluate(model,vl,n).scores()
        hist["train_loss"].append(run/max(nb,1)); hist["val_mIoU"].append(sc["mIoU"]); hist["val_fgIoU"].append(sc["fg_IoU"])
        print(f"[{name}] ep{ep+1}/{epochs} loss={run/max(nb,1):.3f} val_mIoU={sc['mIoU']:.4f} val_fgIoU={sc['fg_IoU']:.4f}")
        if sc["mIoU"]>best+1e-4: best=sc["mIoU"]; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=patience: print(f"  early stop @ ep{ep+1}"); break
    model.load_state_dict(best_state); torch.save(best_state,OUTDIR/f"{name}.pt")
    return model,hist,bs
def deployment(model,img=IMG_SIZE,nw=3,it=15):
    p=sum(x.numel() for x in model.parameters()); tmp=OUTDIR/"_t.pt"; torch.save(model.state_dict(),tmp)
    sz=os.path.getsize(tmp)/1e6; os.remove(tmp); d=torch.randn(1,3,img,img)
    def t(dev):
        m=model.to(dev).eval(); dd=d.to(dev)
        with torch.no_grad():
            for _ in range(nw): m(dd)
            if dev.type=="cuda": torch.cuda.synchronize()
            t0=time.time()
            for _ in range(it): m(dd)
            if dev.type=="cuda": torch.cuda.synchronize()
            return (time.time()-t0)/it*1000
    lg=t(device) if device.type=="cuda" else np.nan; lc=t(torch.device("cpu")); model.to(device)
    return {"params_M":p/1e6,"ckpt_MB":sz,"latency_cpu_ms":lc,"latency_gpu_ms":lg}
def per_image_fg_iou(model):
    model.eval(); v=[]
    with torch.no_grad():
        for r in test_df.itertuples():
            img=np.array(Image.open(r.filepath).convert("RGB")); g=np.array(Image.open(r.mask))
            a=eval_tf(image=img,mask=g); pr=model(a["image"].unsqueeze(0).to(device)).argmax(1)[0].cpu().numpy()
            gg=a["mask"].numpy()>0; pp=pr>0; inter=(gg&pp).sum(); uni=(gg|pp).sum(); v.append(inter/uni if uni>0 else 1.0)
    return np.array(v)

**Run + Save** — trains the model, evaluates on the test split, benchmarks deployment cost, and pickles the full result bundle to `s2_result_<MODEL_NAME>.pkl`:

In [ ]:
# ============ RUN + SAVE ============
t0=time.time(); model,hist,bs=train_one(MODEL_NAME); ttime=time.time()-t0
cf=evaluate(model,loaders(bs)[2],NUM_SEG); sc,iou=cf.scores(); dep=deployment(model)
pifg=per_image_fg_iou(model)
res={"model_name":MODEL_NAME,"seg_task":SEG_TASK,"history":hist,"test_scores":sc,"iou":iou.tolist(),
     "deployment":dep,"per_image_fg_iou":pifg,"img_size":IMG_SIZE,"train_time_sec":ttime}
pickle.dump(res,open(OUTDIR/f"s2_result_{MODEL_NAME}.pkl","wb"))
print("SAVED",OUTDIR/f"s2_result_{MODEL_NAME}.pkl")
print("TEST:",{k:round(v,4) for k,v in sc.items()},"| params",round(dep['params_M'],1),"M | gpu",round(dep['latency_gpu_ms'],1),"ms")

**Per-model figures** — training curve (loss + val mIoU/fg-IoU) and qualitative input/ground-truth/prediction grid, saved to `figures/`:

In [ ]:
# ============ PER-MODEL FIGURES ============
ep=range(1,len(hist["train_loss"])+1)
fig,ax=plt.subplots(figsize=(8,4.5)); ax.plot(ep,hist["train_loss"],"--",label="train loss")
a2=ax.twinx(); a2.plot(ep,hist["val_mIoU"],color="green",label="val mIoU"); a2.plot(ep,hist["val_fgIoU"],color="orange",label="val fg-IoU")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); a2.set_ylabel("IoU"); ax.set_title(f"{MODEL_NAME} — training")
plt.tight_layout(); plt.savefig(FIG/f"{MODEL_NAME}_training.png",dpi=200,bbox_inches="tight"); plt.show()

samp=test_df.sample(min(5,len(test_df)),random_state=SEED).reset_index(drop=True)
fig,axes=plt.subplots(len(samp),3,figsize=(10,3.2*len(samp))); axes=np.atleast_2d(axes)
for i,r in samp.iterrows():
    img=Image.open(r.filepath).convert("RGB"); gt=np.array(Image.open(r["mask"]))>0
    a=eval_tf(image=np.array(img),mask=np.array(Image.open(r["mask"])))
    with torch.no_grad(): pr=model(a["image"].unsqueeze(0).to(device)).argmax(1)[0].cpu().numpy()>0
    axes[i,0].imshow(img.resize((IMG_SIZE,IMG_SIZE))); axes[i,1].imshow(gt,cmap="gray"); axes[i,2].imshow(pr,cmap="gray")
    if i==0:
        for j,t_ in enumerate(["input","ground truth","prediction"]): axes[i,j].set_title(t_)
    axes[i,0].set_ylabel(r.disease,fontsize=8)
    for a_ in axes[i]: a_.set_xticks([]); a_.set_yticks([])
plt.suptitle(f"{MODEL_NAME} — qualitative (lesion mask)"); plt.tight_layout()
plt.savefig(FIG/f"{MODEL_NAME}_qualitative.png",dpi=200,bbox_inches="tight"); plt.show()

---
## End of Session 1

All 3 models in this session are trained and saved to `/kaggle/working/s2_results/`:
- `s2_result_UNet_resnet34.pkl`
- `s2_result_UNetPP_resnet50.pkl`
- `s2_result_DeepLabV3Plus_resnet50.pkl`

**Next step:** download these 3 files. Once Session 2 is also done, combine all 6 `.pkl` files into a single Kaggle Dataset and run the separate **Aggregator** notebook.